# Machine Learning Exam Study Guide
## IIC2115 - Programación como Herramienta para la Ingeniería

**Student Number:** 24641308

---

## How to Use This Notebook

1. **Read the explanations** in markdown cells
2. **Run the code cells** to see examples
3. **Modify variable names** to match your exam data
4. **Practice the workflows** with the exercise datasets

---

# SECTION 0: Core Distinctions (Start Here!)

## Supervised vs Unsupervised Learning

| Supervised Learning | Unsupervised Learning |
|---------------------|----------------------|
| **Has a target variable** (y) to predict | **No target variable** - discover structure |
| Examples: predict species, predict price | Examples: find customer groups, compress data |
| Split data: train/val/test | Usually use all data together |
| Metrics: accuracy, MSE, MAE | Metrics: silhouette score, inertia, variance explained |

## Regression vs Classification (Both are Supervised!)

**Regression (Regresión)**
- **Target variable:** Continuous number
- **Examples:** PM2.5 value (23.4, 45.7...), house value ($150,000, $325,000...)
- **Models:** Linear Regression, Random Forest Regressor, KNN Regressor
- **Metrics:** MSE, MAE, MAPE

**Classification (Clasificación)**
- **Target variable:** Category/label
- **Examples:** species (Adelie, Gentoo...), risk (bajo, alto...), outcome (abandono, graduado...)
- **Models:** Logistic Regression, Random Forest Classifier, KNN Classifier, SVM
- **Metrics:** Accuracy, Balanced Accuracy

## Dimensionality Reduction vs Clustering (Both are Unsupervised!)

**Dimensionality Reduction**
- **Goal:** Transform many features → fewer features
- **Input:** Matrix X with n features
- **Output:** Matrix X_reduced with k features (k < n)
- **Methods:** PCA, t-SNE
- **Use when:** Too many features (>10-15) or want to visualize in 2D/3D

**Clustering**
- **Goal:** Group similar observations together
- **Input:** Matrix X with n features
- **Output:** Array of cluster labels [0, 1, 2, 0, 1, ...]
- **Methods:** K-Means, Agglomerative Clustering
- **Use when:** Want to find natural groups/profiles/patterns

**Can you combine them? YES!**
- **Reduce THEN cluster:** PCA to compress, then cluster on reduced space
- **Cluster THEN reduce:** Cluster on original features, then PCA/t-SNE for 2D visualization
- **Do both and compare:** Try clustering on original vs reduced features (common in exams)

---

# Library Imports

In [ ]:
# Librerías esenciales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

# Modelos supervisados
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.svm import SVC

# Modelos no supervisados
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering

# Métricas
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    accuracy_score, balanced_accuracy_score,
    confusion_matrix, classification_report,
    silhouette_score
)

# Configuración de visualización
%matplotlib inline
sns.set_style('whitegrid')

---

# SECTION 1: Data Preparation

## 1.1 Load and Inspect Data

In [ ]:
# Cargo los datos desde CSV
df = pd.read_csv('datos.csv')

# Inspecciono la estructura del dataset
print(f"Dimensiones: {df.shape}")
print(f"\nPrimeras filas:")
display(df.head())

print(f"\nTipos de datos:")
display(df.dtypes)

print(f"\nEstadísticas descriptivas:")
display(df.describe())

print(f"\nValores faltantes:")
display(df.isnull().sum())

## 1.2 Handle Missing Values

### Strategy:
- **Numerical:** Impute with median (robust to outliers)
- **Categorical:** Drop rows (safer than mode imputation)
- **Timing:** After train/test split to avoid data leakage

In [ ]:
# Identifico columnas numéricas y categóricas
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Si hay columnas categóricas con NaN, las elimino primero
if any(df[categorical_cols].isnull().sum() > 0):
    df = df.dropna(subset=categorical_cols)
    print(f"Filas eliminadas por NaN categóricos. Nueva dimensión: {df.shape}")

## 1.3 Encode Categorical Variables

### Rules:
- **Target variable:** LabelEncoder (safe before split)
- **Features:** OneHotEncoder with `drop_first=True` (avoid multicollinearity)

In [ ]:
# Codifico la variable objetivo (si es clasificación)
target_col = 'species'  # Cambiar según el ejercicio

if target_col in df.columns and df[target_col].dtype == 'object':
    label_encoder = LabelEncoder()
    df[target_col] = label_encoder.fit_transform(df[target_col])
    print(f"Clases codificadas: {label_encoder.classes_}")

In [ ]:
# Codifico variables categóricas predictoras con One-Hot Encoding
categorical_features = ['ocean_proximity']  # Ajustar según ejercicio

if categorical_features:
    df = pd.get_dummies(df, columns=categorical_features, drop_first=True)
    print(f"Nuevas dimensiones después de encoding: {df.shape}")

## 1.4 Feature Engineering

Create derived variables that better represent the problem.

In [ ]:
# Ejemplo: Variables derivadas para housing dataset
# Creo ratios que normalizan por número de hogares

df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']

print("Variables derivadas creadas.")
display(df[['rooms_per_household', 'bedrooms_per_room', 'population_per_household']].head())

In [ ]:
# Ejemplo: Variables derivadas para student dataset
# Creo tasas de aprobación y rendimiento

df['tasa_aprobacion'] = df['asignaturas_aprobadas'] / df['asignaturas_inscritas']
df['tasa_evaluacion'] = df['evaluaciones_realizadas'] / df['asignaturas_inscritas']
df['promedio_ajustado'] = df['promedio_notas'] / df['asignaturas_aprobadas'].replace(0, 1)

# Reemplazo NaN e infinitos resultantes de divisiones por cero
df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

print("Variables de desempeño creadas.")

---

# SECTION 2: Train/Validation/Test Split

## Split Strategy: 60% Train / 20% Validation / 20% Test

**Critical:** Use `stratify` for classification to maintain class proportions.

In [ ]:
# Defino features y target
target = 'species'  # Cambiar según ejercicio
features = [col for col in df.columns if col != target]

X = df[features]
y = df[target]

In [ ]:
# Primera división: 60% train, 40% temp (que luego divido en val/test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=24641308,
    stratify=y  # Solo para clasificación
)

# Segunda división: divido temp en 50/50 para val y test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=24641308,
    stratify=y_temp  # Solo para clasificación
)

print(f"Train: {X_train.shape[0]} ejemplos")
print(f"Validation: {X_val.shape[0]} ejemplos")
print(f"Test: {X_test.shape[0]} ejemplos")

## Handle Missing Values After Split

**Important:** Fit the imputer only on training data to avoid leakage.

In [ ]:
# Calculo la mediana solo del conjunto de entrenamiento
medians = X_train.median()

# Aplico la misma mediana a train, val y test
X_train = X_train.fillna(medians)
X_val = X_val.fillna(medians)
X_test = X_test.fillna(medians)

print("Imputación completada sin data leakage.")

---

# SECTION 3: Feature Scaling

**Critical:** Fit scaler only on training data, then transform all sets.

In [ ]:
# Ajusto el scaler solo con entrenamiento para evitar data leakage
scaler = StandardScaler()
scaler.fit(X_train)

# Transformo train, val y test con los mismos parámetros
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convierto de vuelta a DataFrame para conservar nombres de columnas
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("Escalamiento completado.")
display(X_train_scaled.head())

---

# SECTION 4: Supervised Learning - Regression

## 4.1 Train Multiple Regression Models

In [ ]:
# Defino los modelos de regresión a comparar
regression_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=24641308),
    'KNN': KNeighborsRegressor(n_neighbors=5)
}

# Entreno y evalúo cada modelo en validación
regression_results = []

for name, model in regression_models.items():
    # Entreno con train
    model.fit(X_train_scaled, y_train)
    
    # Predigo en validación
    y_val_pred = model.predict(X_val_scaled)
    
    # Calculo métricas
    mse = mean_squared_error(y_val, y_val_pred)
    mae = mean_absolute_error(y_val, y_val_pred)
    mape = np.mean(np.abs((y_val - y_val_pred) / y_val)) * 100
    
    regression_results.append({
        'Modelo': name,
        'MSE': mse,
        'MAE': mae,
        'MAPE': mape
    })

# Muestro resultados ordenados por MAE
results_df = pd.DataFrame(regression_results).sort_values('MAE')
display(results_df)

## 4.2 Hyperparameter Tuning (Example: Random Forest)

In [ ]:
# Pruebo diferentes configuraciones de Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30]
}

tuning_results = []

for n_est in param_grid['n_estimators']:
    for depth in param_grid['max_depth']:
        # Entreno modelo con estos hiperparámetros
        model = RandomForestRegressor(
            n_estimators=n_est,
            max_depth=depth,
            random_state=24641308
        )
        model.fit(X_train_scaled, y_train)
        
        # Evalúo en validación
        y_val_pred = model.predict(X_val_scaled)
        mae = mean_absolute_error(y_val, y_val_pred)
        
        tuning_results.append({
            'n_estimators': n_est,
            'max_depth': depth,
            'MAE_val': mae
        })

# Encuentro la mejor configuración
tuning_df = pd.DataFrame(tuning_results).sort_values('MAE_val')
display(tuning_df.head())

best_params = tuning_df.iloc[0]
print(f"\nMejor configuración: n_estimators={int(best_params['n_estimators'])}, max_depth={best_params['max_depth']}")

## 4.3 Final Model Evaluation on Test Set

**Only use test set ONCE at the end!**

In [ ]:
# Entreno el modelo final con los mejores hiperparámetros
final_model = RandomForestRegressor(
    n_estimators=int(best_params['n_estimators']),
    max_depth=best_params['max_depth'] if pd.notna(best_params['max_depth']) else None,
    random_state=24641308
)

final_model.fit(X_train_scaled, y_train)

# Evalúo UNA SOLA VEZ en test
y_test_pred = final_model.predict(X_test_scaled)

test_mse = mean_squared_error(y_test, y_test_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mape = np.mean(np.abs((y_test - y_test_pred) / y_test)) * 100

print(f"Resultados finales en Test:")
print(f"MSE: {test_mse:.4f}")
print(f"MAE: {test_mae:.4f}")
print(f"MAPE: {test_mape:.2f}%")

---

# SECTION 5: Supervised Learning - Classification

## 5.1 Train Multiple Classification Models

In [ ]:
# Defino los modelos de clasificación a comparar
classification_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=24641308),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=24641308),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(random_state=24641308)
}

# Entreno y evalúo cada modelo en validación
classification_results = []

for name, model in classification_models.items():
    # Entreno con train
    model.fit(X_train_scaled, y_train)
    
    # Predigo en validación
    y_val_pred = model.predict(X_val_scaled)
    
    # Calculo métricas
    acc = accuracy_score(y_val, y_val_pred)
    bal_acc = balanced_accuracy_score(y_val, y_val_pred)
    
    classification_results.append({
        'Modelo': name,
        'Accuracy': acc,
        'Balanced Accuracy': bal_acc
    })

# Muestro resultados ordenados por Balanced Accuracy
results_df = pd.DataFrame(classification_results).sort_values('Balanced Accuracy', ascending=False)
display(results_df)

## 5.2 Hyperparameter Tuning (Example: Random Forest)

In [ ]:
# Pruebo diferentes configuraciones de Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20]
}

tuning_results = []

for n_est in param_grid['n_estimators']:
    for depth in param_grid['max_depth']:
        # Entreno modelo con estos hiperparámetros
        model = RandomForestClassifier(
            n_estimators=n_est,
            max_depth=depth,
            random_state=24641308
        )
        model.fit(X_train_scaled, y_train)
        
        # Evalúo en validación
        y_val_pred = model.predict(X_val_scaled)
        bal_acc = balanced_accuracy_score(y_val, y_val_pred)
        
        tuning_results.append({
            'n_estimators': n_est,
            'max_depth': depth,
            'Balanced_Accuracy_val': bal_acc
        })

# Encuentro la mejor configuración
tuning_df = pd.DataFrame(tuning_results).sort_values('Balanced_Accuracy_val', ascending=False)
display(tuning_df.head())

best_params = tuning_df.iloc[0]
print(f"\nMejor configuración: n_estimators={int(best_params['n_estimators'])}, max_depth={best_params['max_depth']}")

## 5.3 Final Model Evaluation on Test Set

In [ ]:
# Entreno el modelo final con los mejores hiperparámetros
final_model = RandomForestClassifier(
    n_estimators=int(best_params['n_estimators']),
    max_depth=best_params['max_depth'] if pd.notna(best_params['max_depth']) else None,
    random_state=24641308
)

final_model.fit(X_train_scaled, y_train)

# Evalúo UNA SOLA VEZ en test
y_test_pred = final_model.predict(X_test_scaled)

test_acc = accuracy_score(y_test, y_test_pred)
test_bal_acc = balanced_accuracy_score(y_test, y_test_pred)

print(f"Resultados finales en Test:")
print(f"Accuracy: {test_acc:.4f}")
print(f"Balanced Accuracy: {test_bal_acc:.4f}")

# Matriz de confusión
print(f"\nMatriz de Confusión:")
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

# Reporte de clasificación
print(f"\nReporte de Clasificación:")
print(classification_report(y_test, y_test_pred))

## 5.4 Confusion Matrix Visualization

In [ ]:
# Visualizo la matriz de confusión
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión - Conjunto de Test')
plt.ylabel('Clase Real')
plt.xlabel('Clase Predicha')
plt.show()

---

# SECTION 6: Unsupervised Learning - Dimensionality Reduction

## 6.1 PCA (Principal Component Analysis)

**For unsupervised tasks:** Use all data together (no train/test split needed)

In [ ]:
# Escalo los datos para PCA (sensible a escalas)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Aplico PCA para reducir a 2 componentes (visualización)
pca = PCA(n_components=2, random_state=24641308)
X_pca = pca.fit_transform(X_scaled)

# Analizo la varianza explicada
print(f"Varianza explicada por componente: {pca.explained_variance_ratio_}")
print(f"Varianza total explicada: {pca.explained_variance_ratio_.sum():.2%}")

# Creo DataFrame con componentes principales
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
display(pca_df.head())

## 6.2 PCA Component Interpretation

In [ ]:
# Analizo qué variables contribuyen más a cada componente
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=X.columns
)

print("Contribución de cada variable a las componentes principales:")
display(loadings.sort_values('PC1', key=abs, ascending=False))

## 6.3 PCA Visualization

In [ ]:
# Visualizo los datos en el espacio reducido
plt.figure(figsize=(10, 6))
plt.scatter(pca_df['PC1'], pca_df['PC2'], alpha=0.6)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} varianza)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} varianza)')
plt.title('Proyección PCA (2 componentes)')
plt.grid(True, alpha=0.3)
plt.show()

## 6.4 t-SNE (for Visualization)

**Note:** t-SNE is slower than PCA. Use PCA first if you have many features (>50).

In [ ]:
# Aplico t-SNE para visualización (más lento que PCA)
tsne = TSNE(n_components=2, perplexity=30, random_state=24641308)
X_tsne = tsne.fit_transform(X_scaled)

# Creo DataFrame
tsne_df = pd.DataFrame(X_tsne, columns=['Dim1', 'Dim2'])

# Visualizo
plt.figure(figsize=(10, 6))
plt.scatter(tsne_df['Dim1'], tsne_df['Dim2'], alpha=0.6)
plt.xlabel('Dimensión 1')
plt.ylabel('Dimensión 2')
plt.title('Proyección t-SNE (2 componentes)')
plt.grid(True, alpha=0.3)
plt.show()

---

# SECTION 7: Unsupervised Learning - Clustering

## 7.1 K-Means Clustering

### Step 1: Find Optimal K (Elbow Method)

In [ ]:
# Pruebo diferentes valores de K y calculo la inercia
inertias = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=24641308, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Visualizo el método del codo
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, marker='o', linewidth=2)
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia')
plt.title('Método del Codo para K-Means')
plt.xticks(K_range)
plt.grid(True, alpha=0.3)
plt.show()

print("El 'codo' indica el K óptimo donde la inercia deja de decrecer significativamente.")

### Step 2: Find Optimal K (Silhouette Score)

In [ ]:
# Calculo el Silhouette Score para cada K
silhouette_scores = []
K_range_silhouette = range(2, 11)  # Silhouette requiere al menos 2 clusters

for k in K_range_silhouette:
    kmeans = KMeans(n_clusters=k, random_state=24641308, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

# Visualizo los scores
plt.figure(figsize=(10, 6))
plt.plot(K_range_silhouette, silhouette_scores, marker='o', linewidth=2, color='green')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score para diferentes K')
plt.xticks(K_range_silhouette)
plt.grid(True, alpha=0.3)
plt.show()

# Encuentro el K óptimo
optimal_k = K_range_silhouette[np.argmax(silhouette_scores)]
print(f"K óptimo según Silhouette Score: {optimal_k}")
print(f"Silhouette Score máximo: {max(silhouette_scores):.4f}")

### Step 3: Apply K-Means with Optimal K

In [ ]:
# Aplico K-Means con el K óptimo
optimal_k = 3  # Cambiar según resultados anteriores

kmeans = KMeans(n_clusters=optimal_k, random_state=24641308, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# Agrego las etiquetas al DataFrame original
df['cluster'] = cluster_labels

print(f"Distribución de clusters:")
print(df['cluster'].value_counts().sort_index())

## 7.2 Cluster Visualization with PCA

In [ ]:
# Visualizo los clusters en el espacio PCA
plt.figure(figsize=(10, 6))
scatter = plt.scatter(pca_df['PC1'], pca_df['PC2'], c=cluster_labels, cmap='viridis', alpha=0.6)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} varianza)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} varianza)')
plt.title(f'Clusters K-Means (K={optimal_k}) en espacio PCA')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.show()

## 7.3 Cluster Characterization

In [ ]:
# Caracterizo cada cluster calculando promedios de las variables originales
cluster_profiles = df.groupby('cluster')[X.columns].mean()

print("Perfil promedio de cada cluster:")
display(cluster_profiles)

# Identifico las variables que más diferencian a los clusters
cluster_std = df.groupby('cluster')[X.columns].mean().std()
top_discriminating = cluster_std.sort_values(ascending=False).head(5)

print("\nVariables que más diferencian los clusters:")
print(top_discriminating)

## 7.4 Compare Clustering on Original vs PCA Features

In [ ]:
# Clustering en features originales
kmeans_original = KMeans(n_clusters=optimal_k, random_state=24641308, n_init=10)
labels_original = kmeans_original.fit_predict(X_scaled)
silhouette_original = silhouette_score(X_scaled, labels_original)

# Clustering en features PCA
kmeans_pca = KMeans(n_clusters=optimal_k, random_state=24641308, n_init=10)
labels_pca = kmeans_pca.fit_predict(X_pca)
silhouette_pca = silhouette_score(X_pca, labels_pca)

# Comparo resultados
comparison = pd.DataFrame({
    'Representación': ['Features Originales', 'PCA (2 componentes)'],
    'Silhouette Score': [silhouette_original, silhouette_pca]
})

print("Comparación de clustering en diferentes representaciones:")
display(comparison)

if silhouette_original > silhouette_pca:
    print("\nConclusión: Las features originales producen clusters mejor definidos.")
else:
    print("\nConclusión: La reducción PCA mejora la separación de clusters.")

---

# SECTION 8: Error Analysis

## 8.1 Error Analysis by Cluster/Group

In [ ]:
# Analizo el desempeño del modelo supervisado por cluster
# (Asumiendo que ya tenemos y_test_pred del modelo final)

# Agrego clusters al conjunto de test
test_clusters = kmeans.predict(X_test_scaled)

# Creo DataFrame para análisis
error_analysis = pd.DataFrame({
    'cluster': test_clusters,
    'y_true': y_test.values,
    'y_pred': y_test_pred
})

# Para clasificación: calculo accuracy por cluster
accuracy_by_cluster = error_analysis.groupby('cluster').apply(
    lambda x: accuracy_score(x['y_true'], x['y_pred'])
)

print("Accuracy por cluster:")
print(accuracy_by_cluster)

# Visualizo
plt.figure(figsize=(10, 6))
accuracy_by_cluster.plot(kind='bar', color='skyblue')
plt.xlabel('Cluster')
plt.ylabel('Accuracy')
plt.title('Desempeño del Modelo por Cluster')
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.grid(True, alpha=0.3, axis='y')
plt.show()

## 8.2 Feature Importance (for Tree-based Models)

In [ ]:
# Analizo qué features son más importantes para el modelo
# (Solo funciona con modelos basados en árboles: Random Forest, Decision Tree)

if hasattr(final_model, 'feature_importances_'):
    importance = pd.DataFrame({
        'feature': X.columns,
        'importance': final_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Visualizo top 10 features
    plt.figure(figsize=(10, 6))
    top_features = importance.head(10)
    plt.barh(top_features['feature'], top_features['importance'])
    plt.xlabel('Importancia')
    plt.title('Top 10 Features Más Importantes')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("Top 10 features más importantes:")
    display(top_features)
else:
    print("Este modelo no proporciona feature importances.")

---

# SECTION 9: E7-Style Integrated Workflow

## Complete workflow for exercises combining supervised + unsupervised learning

### Phase 1: Prepare Two Feature Matrices

In [ ]:
# Ejemplo: E7 Student trajectories

# Features disponibles al ingreso
ingreso_features = ['programa', 'orden_preferencia', 'jornada', 'nota_admision',
                    'moroso', 'arancel_al_dia', 'beca', 'edad_ingreso',
                    'desempleo', 'inflacion', 'crecimiento_pib']

# Features que incluyen desempeño académico
actualizada_features = ingreso_features + [
    'asignaturas_convalidadas', 'asignaturas_inscritas',
    'evaluaciones_realizadas', 'asignaturas_aprobadas',
    'promedio_notas', 'tasa_aprobacion', 'tasa_evaluacion'
]

# Creo las dos matrices
X_ingreso = df[ingreso_features]
X_actualizada = df[actualizada_features]

print(f"Matriz de ingreso: {X_ingreso.shape}")
print(f"Matriz actualizada: {X_actualizada.shape}")

### Phase 2: Unsupervised Analysis (Profiles)

Use ONLY performance features (not outcome, not ingreso features)

In [ ]:
# Features solo de desempeño (para clustering)
desempeno_features = ['asignaturas_convalidadas', 'asignaturas_inscritas',
                      'evaluaciones_realizadas', 'asignaturas_aprobadas',
                      'promedio_notas', 'tasa_aprobacion', 'tasa_evaluacion']

X_desempeno = df[desempeno_features]

# Escalo
scaler_desempeno = StandardScaler()
X_desempeno_scaled = scaler_desempeno.fit_transform(X_desempeno)

# PCA para visualización
pca_desempeno = PCA(n_components=2, random_state=24641308)
X_desempeno_pca = pca_desempeno.fit_transform(X_desempeno_scaled)

# Clustering
kmeans_desempeno = KMeans(n_clusters=3, random_state=24641308, n_init=10)
perfiles = kmeans_desempeno.fit_predict(X_desempeno_scaled)

df['perfil'] = perfiles

print(f"Distribución de perfiles de desempeño:")
print(df['perfil'].value_counts().sort_index())

### Phase 3: Supervised Prediction on Both Matrices

In [ ]:
# Target
target = 'resultado'
y = df[target]

# Split (mismo split para ambas matrices)
X_ingreso_train, X_ingreso_temp, y_train, y_temp = train_test_split(
    X_ingreso, y, test_size=0.40, random_state=24641308, stratify=y
)

X_ingreso_val, X_ingreso_test, y_val, y_test = train_test_split(
    X_ingreso_temp, y_temp, test_size=0.50, random_state=24641308, stratify=y_temp
)

# Aplico el mismo split a X_actualizada
X_actualizada_train = X_actualizada.loc[X_ingreso_train.index]
X_actualizada_val = X_actualizada.loc[X_ingreso_val.index]
X_actualizada_test = X_actualizada.loc[X_ingreso_test.index]

# Escalo ambas matrices
scaler_ingreso = StandardScaler()
X_ingreso_train_scaled = scaler_ingreso.fit_transform(X_ingreso_train)
X_ingreso_val_scaled = scaler_ingreso.transform(X_ingreso_val)
X_ingreso_test_scaled = scaler_ingreso.transform(X_ingreso_test)

scaler_actualizada = StandardScaler()
X_actualizada_train_scaled = scaler_actualizada.fit_transform(X_actualizada_train)
X_actualizada_val_scaled = scaler_actualizada.transform(X_actualizada_val)
X_actualizada_test_scaled = scaler_actualizada.transform(X_actualizada_test)

print("Matrices preparadas para modelamiento supervisado.")

In [ ]:
# Entreno modelos en AMBAS matrices
model = RandomForestClassifier(n_estimators=100, random_state=24641308)

# Modelo solo con features de ingreso
model.fit(X_ingreso_train_scaled, y_train)
y_pred_ingreso = model.predict(X_ingreso_test_scaled)
bal_acc_ingreso = balanced_accuracy_score(y_test, y_pred_ingreso)

# Modelo con features actualizadas
model.fit(X_actualizada_train_scaled, y_train)
y_pred_actualizada = model.predict(X_actualizada_test_scaled)
bal_acc_actualizada = balanced_accuracy_score(y_test, y_pred_actualizada)

# Comparo
print(f"Balanced Accuracy solo con ingreso: {bal_acc_ingreso:.4f}")
print(f"Balanced Accuracy con desempeño: {bal_acc_actualizada:.4f}")
print(f"Mejora absoluta: {bal_acc_actualizada - bal_acc_ingreso:.4f}")

### Phase 4: Performance Analysis by Profile

In [ ]:
# Asigno perfiles al conjunto de test
test_perfiles = df.loc[X_ingreso_test.index, 'perfil']

# Analizo balanced accuracy por perfil
for perfil in sorted(test_perfiles.unique()):
    mask = test_perfiles == perfil
    
    bal_acc_ingreso_perfil = balanced_accuracy_score(
        y_test[mask], y_pred_ingreso[mask]
    )
    bal_acc_actualizada_perfil = balanced_accuracy_score(
        y_test[mask], y_pred_actualizada[mask]
    )
    mejora = bal_acc_actualizada_perfil - bal_acc_ingreso_perfil
    
    print(f"\nPerfil {perfil}:")
    print(f"  Bal. Acc. ingreso: {bal_acc_ingreso_perfil:.4f}")
    print(f"  Bal. Acc. actualizada: {bal_acc_actualizada_perfil:.4f}")
    print(f"  Mejora: {mejora:.4f}")

---

# SECTION 10: Key Metrics Reference

## Regression Metrics

In [ ]:
# Mean Squared Error (MSE)
mse = mean_squared_error(y_true, y_pred)

# Mean Absolute Error (MAE) - más interpretable
mae = mean_absolute_error(y_true, y_pred)

# Mean Absolute Percentage Error (MAPE)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.2f}%")

## Classification Metrics

In [ ]:
# Accuracy (simple)
acc = accuracy_score(y_true, y_pred)

# Balanced Accuracy (mejor para clases desbalanceadas)
bal_acc = balanced_accuracy_score(y_true, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")

# Usa Balanced Accuracy para decidir el mejor modelo si hay desbalance

## Clustering Metrics

In [ ]:
# Silhouette Score ([-1, 1], más alto es mejor)
sil_score = silhouette_score(X_scaled, cluster_labels)

# Inertia (más bajo es mejor, pero siempre decrece con más K)
inertia = kmeans.inertia_

print(f"Silhouette Score: {sil_score:.4f}")
print(f"Inertia: {inertia:.2f}")

---

# SECTION 11: Common Pitfalls

## ❌ DON'T

```python
# ❌ Fit scaler on entire dataset (data leakage)
scaler.fit(X)  

# ❌ Use test set multiple times
for config in configs:
    score = model.score(X_test, y_test)

# ❌ Include target in clustering features
X_cluster = df[['feature1', 'feature2', 'target']]  

# ❌ Forget stratify in classification split
train_test_split(X, y, test_size=0.2)  
```

## ✅ DO

```python
# ✅ Fit scaler only on training
scaler.fit(X_train)

# ✅ Use validation for model selection, test only once
for config in configs:
    score = model.score(X_val, y_val)
final_score = best_model.score(X_test, y_test)  # Only once!

# ✅ Exclude target from clustering
X_cluster = df[['feature1', 'feature2']]  

# ✅ Use stratify for balanced splits
train_test_split(X, y, test_size=0.2, stratify=y)  
```

---

# Final Exam Tips

1. **Read the exercise carefully** to identify:
   - Is it regression or classification? (continuous vs categorical target)
   - Is it supervised or unsupervised? (have target or not)
   - Which variables can you use for clustering? (exclude target!)

2. **Follow the workflow:**
   - Load → Inspect → Handle missing → Encode → Feature engineering
   - Split (train/val/test) → Scale
   - Train models → Select best on validation
   - Evaluate ONCE on test

3. **For unsupervised:**
   - Scale first (K-Means and PCA are sensitive to scale)
   - Use silhouette score to compare K values
   - Characterize clusters using original variables

4. **Safe model choices:**
   - Regression: Linear Regression + Random Forest
   - Classification: Logistic Regression + Random Forest
   - Clustering: K-Means (always works)
   - Dimensionality reduction: PCA (faster, more interpretable)

5. **Use Balanced Accuracy** for classification with imbalanced classes

6. **Always use `random_state=24641308`** for reproducibility

7. **Comment your code in Spanish** to show you understand what you're doing

---

**¡Buena suerte en el examen! 🎯**